# 🎵 VoxConverse Interactive Dataset Explorer

Ce notebook permet d'explorer interactivement les segments du dataset VoxConverse avec sauvegarde/chargement automatique pour éviter le retraitement des données.

## Fonctionnalités
- 💾 **Sauvegarde automatique** : Le dataset est traité une seule fois et mis en cache
- 🎛️ **Exploration interactive** : Widgets pour naviguer entre les segments
- 📊 **Visualisations complètes** : Spectrogramme mel, signal audio, VAD, OSD, VCN
- 🎧 **Lecture audio** : Écoute directe des segments sélectionnés
- ⏱️ **Contrôle temporel** : Zoom sur des parties spécifiques des segments

## 1. Import des bibliothèques et configuration

In [1]:
# Import des bibliothèques essentielles
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchaudio
from IPython.display import display, Audio, clear_output
import ipywidgets as widgets
from ipywidgets import interactive
import pickle
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

# Ajouter le dossier src au path
sys.path.append('./src')

# Import du dataset VoxConverse
from src.voxconverse_dataset import VoxConverseDataset

# Configuration globale
SEGMENT_DURATION = 30.0    # Durée des segments en secondes (plus court pour l'exploration)
HOP_DURATION = 15.0        # Chevauchement entre segments
SAMPLE_RATE = 16000        # Fréquence d'échantillonnage
N_MELS = 80               # Nombre de bandes mel
SELECTED_SEGMENT_IDX = 0  # Segment initial à afficher
DISPLAY_TIME_RANGE = [0, SEGMENT_DURATION]  # Plage temporelle d'affichage

print("✅ Bibliothèques importées avec succès!")
print(f"📊 Configuration:")
print(f"   Durée des segments: {SEGMENT_DURATION}s")
print(f"   Fréquence d'échantillonnage: {SAMPLE_RATE} Hz")
print(f"   Bandes mel: {N_MELS}")

✅ Bibliothèques importées avec succès!
📊 Configuration:
   Durée des segments: 30.0s
   Fréquence d'échantillonnage: 16000 Hz
   Bandes mel: 80


## 2. Classe DatasetManager pour la sauvegarde/chargement

In [2]:
class DatasetManager:
    """
    Gestionnaire de dataset avec sauvegarde/chargement automatique
    pour éviter de retraiter les données à chaque session.
    """
    
    def __init__(self, cache_dir="./voxconverse_cache"):
        self.cache_dir = Path(cache_dir)
        self.cache_dir.mkdir(exist_ok=True)
        self.dataset = None
        
    def get_cache_filename(self, split, segment_duration, hop_duration, sample_rate, n_mels):
        """Génère un nom de fichier de cache basé sur les paramètres."""
        params = f"{split}_{segment_duration}_{hop_duration}_{sample_rate}_{n_mels}"
        return self.cache_dir / f"voxconverse_explorer_{params}.pkl"
    
    def save_dataset(self, dataset, filename):
        """Sauvegarde le dataset traité."""
        try:
            with open(filename, 'wb') as f:
                pickle.dump({
                    'segments': dataset.segments,
                    'dataset_info': {
                        'split': dataset.split,
                        'segment_duration': dataset.segment_duration,
                        'hop_duration': dataset.hop_duration,
                        'sample_rate': dataset.sample_rate,
                        'n_mels': dataset.n_mels
                    }
                }, f)
            print(f"💾 Dataset sauvegardé: {filename}")
            return True
        except Exception as e:
            print(f"❌ Erreur de sauvegarde: {e}")
            return False
    
    def load_dataset(self, filename):
        """Charge un dataset sauvegardé."""
        try:
            if not filename.exists():
                return None
                
            with open(filename, 'rb') as f:
                data = pickle.load(f)
            
            print(f"📂 Dataset chargé: {filename}")
            print(f"   📦 {len(data['segments'])} segments")
            return data
        except Exception as e:
            print(f"❌ Erreur de chargement: {e}")
            return None
    
    def get_or_create_dataset(self, split='dev', segment_duration=30.0, hop_duration=15.0, 
                             sample_rate=16000, n_mels=80, force_reload=False):
        """
        Charge un dataset existant ou en crée un nouveau si nécessaire.
        """
        cache_filename = self.get_cache_filename(split, segment_duration, hop_duration, sample_rate, n_mels)
        
        # Essayer de charger depuis le cache
        if not force_reload:
            cached_data = self.load_dataset(cache_filename)
            if cached_data:
                # Recréer un objet dataset avec les données chargées
                dataset = VoxConverseDataset(
                    split=split,
                    segment_duration=segment_duration,
                    hop_duration=hop_duration,
                    sample_rate=sample_rate,
                    n_mels=n_mels,
                    cache_dir=str(self.cache_dir),
                    force_reload=False  # Utilise le cache intégré
                )
                return dataset
        
        # Créer un nouveau dataset
        print("🔄 Création d'un nouveau dataset...")
        dataset = VoxConverseDataset(
            split=split,
            segment_duration=segment_duration,
            hop_duration=hop_duration,
            sample_rate=sample_rate,
            n_mels=n_mels,
            cache_dir=str(self.cache_dir),
            force_reload=force_reload
        )
        
        return dataset

# Initialiser le gestionnaire
dataset_manager = DatasetManager()
print("✅ DatasetManager initialisé")

✅ DatasetManager initialisé


## 3. Chargement ou création du dataset

In [3]:
# Charger ou créer le dataset
print("🔍 Vérification du cache et chargement du dataset...")
print("=" * 60)

try:
    # Obtenir le dataset (chargé depuis le cache ou créé)
    dataset = dataset_manager.get_or_create_dataset(
        split='dev',
        segment_duration=SEGMENT_DURATION,
        hop_duration=HOP_DURATION,
        sample_rate=SAMPLE_RATE,
        n_mels=N_MELS,
        force_reload=False  # Changez à True pour forcer le rechargement
    )
    
    print(f"\n✅ Dataset prêt à l'exploration!")
    print(f"📊 Informations du dataset:")
    print(f"   📦 Segments disponibles: {len(dataset)}")
    print(f"   📏 Durée par segment: {SEGMENT_DURATION}s")
    print(f"   🔄 Chevauchement: {HOP_DURATION}s")
    print(f"   🎵 Fréquence: {SAMPLE_RATE} Hz")
    
    # Informations sur le cache
    cache_info = dataset.get_cache_info()
    print(f"\n💾 Informations du cache:")
    print(f"   📁 Répertoire: {cache_info['cache_dir']}")
    print(f"   🗃️  Fichier: {cache_info['cache_file']}")
    print(f"   ✅ Cache existant: {cache_info['cache_exists']}")
    if 'cache_size_mb' in cache_info:
        print(f"   📏 Taille: {cache_info['cache_size_mb']:.1f} MB")
    
    # Exemples de segments
    print(f"\n📋 Aperçu des premiers segments:")
    for i in range(min(3, len(dataset))):
        seg = dataset.segments[i]
        print(f"   {i}: Conv {seg['conv_idx']}, t={seg['start_time']:.1f}-{seg['end_time']:.1f}s")
        
except Exception as e:
    print(f"❌ Erreur lors du chargement: {e}")
    import traceback
    traceback.print_exc()

🔍 Vérification du cache et chargement du dataset...
🔄 Création d'un nouveau dataset...
✅ Loaded dataset from cache: voxconverse_cache/voxconverse_37aa6db9ed28.pkl
📦 4547 segments ready
🎯 Dataset ready: 4547 segments

✅ Dataset prêt à l'exploration!
📊 Informations du dataset:
   📦 Segments disponibles: 4547
   📏 Durée par segment: 30.0s
   🔄 Chevauchement: 15.0s
   🎵 Fréquence: 16000 Hz

💾 Informations du cache:
   📁 Répertoire: voxconverse_cache
   🗃️  Fichier: voxconverse_cache/voxconverse_37aa6db9ed28.pkl
   ✅ Cache existant: True
   📏 Taille: 8404.9 MB

📋 Aperçu des premiers segments:
   0: Conv 0, t=0.0-30.0s
   1: Conv 0, t=15.0-45.0s
   2: Conv 0, t=30.0-60.0s
✅ Loaded dataset from cache: voxconverse_cache/voxconverse_37aa6db9ed28.pkl
📦 4547 segments ready
🎯 Dataset ready: 4547 segments

✅ Dataset prêt à l'exploration!
📊 Informations du dataset:
   📦 Segments disponibles: 4547
   📏 Durée par segment: 30.0s
   🔄 Chevauchement: 15.0s
   🎵 Fréquence: 16000 Hz

💾 Informations du cach

## 4. Fonction de visualisation interactive

In [4]:
def create_interactive_widget(dataset):
    """Crée un widget interactif pour explorer les segments"""
    
    def update_visualization(segment_idx, show_audio_player, time_start, time_end):
        """Fonction appelée quand les contrôles changent"""
        
        # Clear previous output
        clear_output(wait=True)
        
        if segment_idx >= len(dataset):
            print(f"❌ Segment {segment_idx} n'existe pas (max: {len(dataset)-1})")
            return
        
        try:
            # Obtenir le segment
            sample = dataset[segment_idx]
            segment_info = dataset.segments[segment_idx]
            
            # Données
            mel_features = sample['features'].numpy()
            vad_labels = sample['vad_labels'].numpy()
            osd_labels = sample['osd_labels'].numpy()
            vcn_labels = sample['vcn_labels'].numpy()
            audio = sample['audio'].numpy()
            
            # Configuration temporelle
            time_frames = mel_features.shape[1]
            time_axis = np.linspace(0, SEGMENT_DURATION, time_frames)
            audio_time = np.linspace(0, SEGMENT_DURATION, len(audio))
            
            # Créer la figure
            fig, axes = plt.subplots(5, 1, figsize=(16, 12))
            fig.suptitle(f'🎵 Segment #{segment_idx} - Conv {segment_info["conv_idx"]} - '
                         f't={segment_info["start_time"]:.1f}s-{segment_info["end_time"]:.1f}s', 
                         fontsize=14, fontweight='bold')
            
            # 1. Mel Spectrogramme
            im = axes[0].imshow(mel_features, aspect='auto', origin='lower',
                                extent=[0, SEGMENT_DURATION, 0, N_MELS],
                                cmap='viridis')
            axes[0].set_title('🎼 Mel Spectrogramme', fontweight='bold')
            axes[0].set_ylabel('Bandes Mel')
            axes[0].set_xlim(time_start, time_end)
            plt.colorbar(im, ax=axes[0], shrink=0.8)
            
            # 2. Signal audio
            axes[1].plot(audio_time, audio, color='blue', alpha=0.7, linewidth=0.8)
            axes[1].set_title('🔊 Signal Audio', fontweight='bold')
            axes[1].set_ylabel('Amplitude')
            axes[1].set_xlim(time_start, time_end)
            axes[1].grid(True, alpha=0.3)
            
            # 3. VAD
            axes[2].fill_between(time_axis, 0, vad_labels, alpha=0.7, color='green')
            axes[2].plot(time_axis, vad_labels, color='darkgreen', linewidth=2)
            axes[2].set_title('🎤 VAD - Voice Activity', fontweight='bold')
            axes[2].set_ylabel('Activation')
            axes[2].set_xlim(time_start, time_end)
            axes[2].set_ylim(-0.1, 1.1)
            axes[2].grid(True, alpha=0.3)
            
            # 4. OSD
            axes[3].fill_between(time_axis, 0, osd_labels, alpha=0.7, color='orange')
            axes[3].plot(time_axis, osd_labels, color='darkorange', linewidth=2)
            axes[3].set_title('🗣️  OSD - Overlap Speech', fontweight='bold')
            axes[3].set_ylabel('Activation')
            axes[3].set_xlim(time_start, time_end)
            axes[3].set_ylim(-0.1, 1.1)
            axes[3].grid(True, alpha=0.3)
            
            # 5. VCN
            axes[4].fill_between(time_axis, 0, vcn_labels, alpha=0.7, color='red')
            axes[4].plot(time_axis, vcn_labels, color='darkred', linewidth=2)
            axes[4].set_title('🔄 VCN - Voice Change', fontweight='bold')
            axes[4].set_ylabel('Activation')
            axes[4].set_xlabel('Temps (secondes)')
            axes[4].set_xlim(time_start, time_end)
            axes[4].set_ylim(-0.1, 1.1)
            axes[4].grid(True, alpha=0.3)
            
            plt.tight_layout()
            plt.show()
            
            # Statistiques
            print(f"📊 Segment #{segment_idx} | Conv {segment_info['conv_idx']} | {segment_info['start_time']:.1f}s-{segment_info['end_time']:.1f}s")
            print(f"🎤 VAD: {np.sum(vad_labels > 0)}/{len(vad_labels)} ({np.mean(vad_labels)*100:.1f}%) | "
                  f"🗣️  OSD: {np.sum(osd_labels > 0)}/{len(osd_labels)} ({np.mean(osd_labels)*100:.1f}%) | "
                  f"🔄 VCN: {np.sum(vcn_labels > 0)}/{len(vcn_labels)} ({np.mean(vcn_labels)*100:.1f}%)")
            
            # Informations sur les speakers
            speakers = segment_info.get('speaker_ids', [])
            if speakers:
                print(f"👥 Speakers: {speakers}")
            
            # Lecture audio si demandée
            if show_audio_player:
                print("🎧 Lecture audio:")
                # Extraire la partie audio correspondant à la plage temporelle
                start_sample = int(time_start * SAMPLE_RATE)
                end_sample = int(time_end * SAMPLE_RATE)
                audio_segment = audio[start_sample:end_sample]
                display(Audio(audio_segment, rate=SAMPLE_RATE))
                
        except Exception as e:
            print(f"❌ Erreur lors de la visualisation: {e}")
            import traceback
            traceback.print_exc()
    
    return update_visualization

print("✅ Fonction de visualisation créée")

✅ Fonction de visualisation créée


## 5. Initialisation des contrôles interactifs

In [5]:
# Créer la fonction de visualisation
update_viz = create_interactive_widget(dataset)

# Créer les contrôles interactifs
max_segments = len(dataset) - 1

print("🎛️ Création des contrôles interactifs...")

# Sélecteur de segment
segment_input = widgets.IntText(
    value=SELECTED_SEGMENT_IDX,
    min=0,
    max=max_segments,
    description=f'Segment (0-{max_segments}):',
    style={'description_width': 'initial'},
    layout={'width': '250px'}
)

# Checkbox pour la lecture audio
audio_checkbox = widgets.Checkbox(
    value=True,
    description='🎧 Lecture audio',
    style={'description_width': 'initial'},
    layout={'width': '150px'}
)

# Slider pour le début de la plage temporelle
time_start_slider = widgets.FloatSlider(
    value=DISPLAY_TIME_RANGE[0],
    min=0,
    max=SEGMENT_DURATION,
    step=0.1,
    description='⏪ Début (s):',
    style={'description_width': 'initial'},
    layout={'width': '400px'}
)

# Slider pour la fin de la plage temporelle
time_end_slider = widgets.FloatSlider(
    value=DISPLAY_TIME_RANGE[1],
    min=0,
    max=SEGMENT_DURATION,
    step=0.1,
    description='⏩ Fin (s):',
    style={'description_width': 'initial'},
    layout={'width': '400px'}
)

# Boutons de navigation rapide
def create_navigation_buttons():
    """Crée des boutons pour la navigation rapide."""
    
    def go_to_segment(segment_idx):
        """Va directement à un segment spécifique."""
        segment_input.value = max(0, min(segment_idx, max_segments))
    
    prev_button = widgets.Button(
        description='⬅️ Précédent',
        button_style='primary',
        layout={'width': '120px'}
    )
    
    next_button = widgets.Button(
        description='➡️ Suivant',
        button_style='primary', 
        layout={'width': '120px'}
    )
    
    random_button = widgets.Button(
        description='🎲 Aléatoire',
        button_style='warning',
        layout={'width': '120px'}
    )
    
    def on_prev_clicked(b):
        go_to_segment(segment_input.value - 1)
    
    def on_next_clicked(b):
        go_to_segment(segment_input.value + 1)
        
    def on_random_clicked(b):
        import random
        go_to_segment(random.randint(0, max_segments))
    
    prev_button.on_click(on_prev_clicked)
    next_button.on_click(on_next_clicked)
    random_button.on_click(on_random_clicked)
    
    return widgets.HBox([prev_button, next_button, random_button])

# Créer les boutons de navigation
nav_buttons = create_navigation_buttons()

print("✅ Contrôles créés:")
print(f"   📊 {max_segments + 1} segments disponibles")
print(f"   ⏱️  Plage temporelle: 0-{SEGMENT_DURATION}s")
print(f"   🎯 Navigation: boutons et sliders")

🎛️ Création des contrôles interactifs...
✅ Contrôles créés:
   📊 4547 segments disponibles
   ⏱️  Plage temporelle: 0-30.0s
   🎯 Navigation: boutons et sliders


## 6. Widget interactif final - Exploration des segments

In [6]:
# Créer le widget interactif principal
print("🎛️ EXPLORATEUR INTERACTIF VOXCONVERSE")
print("=" * 60)
print(f"📦 {len(dataset)} segments disponibles")
print(f"🎯 Segment initial: #{SELECTED_SEGMENT_IDX}")
print(f"📏 Durée: {SEGMENT_DURATION}s par segment")
print(f"🎧 Lecture audio intégrée")
print()

# Organiser les contrôles en interface
controls_box = widgets.VBox([
    widgets.HTML("<h3>🎛️ Contrôles de Navigation</h3>"),
    widgets.HBox([segment_input, audio_checkbox]),
    nav_buttons,
    widgets.HTML("<h3>⏱️ Contrôle Temporel</h3>"),
    time_start_slider,
    time_end_slider,
    widgets.HTML("<br><i>💡 Utilisez les sliders temporels pour zoomer sur des parties spécifiques du segment</i>")
])

# Créer le widget interactif
interactive_widget = interactive(
    update_viz,
    segment_idx=segment_input,
    show_audio_player=audio_checkbox,
    time_start=time_start_slider,
    time_end=time_end_slider
)

# Interface complète
full_interface = widgets.VBox([
    controls_box,
    widgets.HTML("<hr>"),
    interactive_widget
])

# Afficher l'interface
display(full_interface)

print("✅ Widget interactif prêt!")
print()
print("📚 Instructions d'utilisation:")
print("   1. 📊 Utilisez le champ 'Segment' pour naviguer entre les segments")
print("   2. 🎧 Cochez 'Lecture audio' pour entendre le segment")
print("   3. ⏱️  Ajustez les sliders temporels pour zoomer sur des parties spécifiques")
print("   4. 🎲 Utilisez les boutons pour naviguer rapidement")
print("   5. 📈 Observez les 5 visualisations: Mel, Audio, VAD, OSD, VCN")

🎛️ EXPLORATEUR INTERACTIF VOXCONVERSE
📦 4547 segments disponibles
🎯 Segment initial: #0
📏 Durée: 30.0s par segment
🎧 Lecture audio intégrée



✅ Widget interactif prêt!

📚 Instructions d'utilisation:
   1. 📊 Utilisez le champ 'Segment' pour naviguer entre les segments
   2. 🎧 Cochez 'Lecture audio' pour entendre le segment
   3. ⏱️  Ajustez les sliders temporels pour zoomer sur des parties spécifiques
   4. 🎲 Utilisez les boutons pour naviguer rapidement
   5. 📈 Observez les 5 visualisations: Mel, Audio, VAD, OSD, VCN


---

## 🔧 Utilitaires supplémentaires

Cellules optionnelles pour la gestion avancée du dataset et du cache.

In [ ]:
# 💾 Gestion du cache
print("🔧 UTILITAIRES DE GESTION DU CACHE")
print("=" * 50)

# Informations détaillées sur le cache
cache_info = dataset.get_cache_info()
print("📊 Informations détaillées du cache:")
for key, value in cache_info.items():
    print(f"   {key}: {value}")

print("\n📁 Fichiers de cache disponibles:")
cache_files = VoxConverseDataset.list_cache_files()
for i, file in enumerate(cache_files):
    print(f"   {i+1}. {file}")

# Fonction pour vider le cache si nécessaire
def clear_cache_and_reload():
    """Vide le cache et recharge le dataset."""
    global dataset  # Déclaration globale AVANT utilisation
    dataset.clear_cache()
    dataset = dataset_manager.get_or_create_dataset(
        split='dev',
        segment_duration=SEGMENT_DURATION,
        hop_duration=HOP_DURATION,
        sample_rate=SAMPLE_RATE,
        n_mels=N_MELS,
        force_reload=True
    )
    print("✅ Dataset rechargé avec succès!")

print("\n💡 Pour vider le cache et recharger:")
print("   Exécutez: clear_cache_and_reload()")

🔧 UTILITAIRES DE GESTION DU CACHE
📊 Informations détaillées du cache:
   cache_dir: voxconverse_cache
   cache_file: voxconverse_cache/voxconverse_37aa6db9ed28.pkl
   cache_exists: True
   cache_key: 37aa6db9ed28
   segments_count: 4547
   cache_size_mb: 8404.9159450531

📁 Fichiers de cache disponibles:
   1. voxconverse_cache/voxconverse_37aa6db9ed28.pkl


SyntaxError: name 'dataset' is used prior to global declaration (3522292834.py, line 20)

In [ ]:
# 📊 Statistiques du dataset
print("📊 STATISTIQUES DÉTAILLÉES DU DATASET")
print("=" * 50)

# Analyser la distribution des activités
vad_counts = sum(1 for seg in dataset.segments if seg.get('vad_frames', 0) > 0)
osd_counts = sum(1 for seg in dataset.segments if seg.get('osd_frames', 0) > 0)
vcn_counts = sum(1 for seg in dataset.segments if seg.get('vcn_frames', 0) > 0)

print(f"📈 Distribution des activités:")
print(f"   🎤 Segments avec VAD: {vad_counts}/{len(dataset)} ({vad_counts/len(dataset)*100:.1f}%)")
print(f"   🗣️  Segments avec OSD: {osd_counts}/{len(dataset)} ({osd_counts/len(dataset)*100:.1f}%)")
print(f"   🔄 Segments avec VCN: {vcn_counts}/{len(dataset)} ({vcn_counts/len(dataset)*100:.1f}%)")

# Conversations uniques
conv_ids = set(seg['conv_idx'] for seg in dataset.segments)
print(f"\n📋 Conversations:")
print(f"   💬 Conversations uniques: {len(conv_ids)}")
print(f"   📦 Segments par conversation: {len(dataset)/len(conv_ids):.1f} en moyenne")

# Segments intéressants
print(f"\n🎯 Segments recommandés pour l'exploration:")

# Segments avec beaucoup d'activité OSD
osd_segments = [(i, seg) for i, seg in enumerate(dataset.segments) if seg.get('osd_frames', 0) > 100]
if osd_segments:
    print("   🗣️  Segments avec overlap important:")
    for i, (idx, seg) in enumerate(osd_segments[:3]):
        print(f"      {idx}: Conv {seg['conv_idx']}, OSD={seg.get('osd_frames', 0)} frames")

# Segments avec changements de voix
vcn_segments = [(i, seg) for i, seg in enumerate(dataset.segments) if seg.get('vcn_frames', 0) > 50]
if vcn_segments:
    print("   🔄 Segments avec changements de voix:")
    for i, (idx, seg) in enumerate(vcn_segments[:3]):
        print(f"      {idx}: Conv {seg['conv_idx']}, VCN={seg.get('vcn_frames', 0)} frames")

print(f"\n✅ Prêt pour l'exploration! Utilisez le widget ci-dessus.")